# Reservoir facies classification — well-based machine learning

**Goal:** Predict nine depositional facies from core-calibrated wireline measurements and geological indicators in the **Panoma (Council Grove) gas reservoir, southwest Kansas**, and evaluate generalization to *entirely unseen wells*.

**Geological context:** The Lower Permian Council Grove Group lies in the Hugoton Embayment of the Anadarko Basin, stratigraphically beneath the Hugoton (Chase Group) gas reservoir. Repeated marine incursions and regressions produced mixed carbonate–siliciclastic intervals; lithofacies distinguish sandstone/siltstone/shale, limestone textures, and dolomite.

**Dataset:** 4,149 labeled rows (9 real wells + 80 pseudo-well rows) and 830 unlabeled rows (STUART, CRAWFORD). Five measured well-log curves (GR, ILD_log10, DeltaPHI, PHIND, PE) and two geological descriptors (NM_M, RELPOS). PE is absent in ALEXANDER D and KIMZEY A.

**Protocol:** (1) Clean and audit genuine well observations; (2) use 4 measured logs + 2 geological indicators, retaining all 9 labeled real wells; (3) test all nine wells in turn with **outer leave-one-well-out** evaluation; (4) compare three fixed models via grouped inner CV (no grid search); (5) compute exact/adjacent metrics and geology-sensitive plots; (6) train the final model on all labeled wells and predict 2 unlabeled wells.

**Reading guide:** Sections 1–5 cover the dataset, preprocessing, cross-well evaluation and feature comparisons. Section 6 retains the original-style five-log + facies tracks for **CHURCHMAN BIBLE** and **LUKE G U**; sections 7–8 show final inference and limitations. See the [README](README.md) for detailed geological setting, source attribution and results.

In [1]:
from pathlib import Path
import json, time, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from sklearn import __version__ as SKLEARN_VERSION
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneGroupOut, GroupKFold
from sklearn.metrics import (accuracy_score, f1_score, balanced_accuracy_score, confusion_matrix,
                             classification_report)
from sklearn.inspection import permutation_importance

SEED=42
ROOT=Path.cwd()
OUT=ROOT/'outputs';OUT.mkdir(exist_ok=True)
LOGS_4=['GR','ILD_log10','DeltaPHI','PHIND']
LOGS_5=LOGS_4+['PE']
GEO=['NM_M','RELPOS']
PRIMARY_FEATURES=LOGS_4+GEO
PE_FEATURES=LOGS_5+GEO
FACIES={1:'SS',2:'CSiS',3:'FSiS',4:'SiSh',5:'MS',6:'WS',7:'D',8:'PS',9:'BS'}
COLORS=['#F4D03F','#F5B041','#DC7633','#6E2C00','#1B4F72','#2E86C1','#AED6F1','#A569BD','#196F3D']
ADJACENT={1:{2},2:{1,3},3:{2},4:{5},5:{4,6},6:{5,7,8},7:{6,8},8:{6,7,9},9:{7,8}}
print('scikit-learn:',SKLEARN_VERSION,'| folder:',ROOT)
# Preserve the complete original CSVs; all cleaning happens in copies in memory.

scikit-learn: 1.8.0 | folder: /mnt/data/Reservoir_Facies_Classification_Well_Validated


## 1. Data integrity and well coverage

The labeled source includes a recruited **Recruit F9 pseudo-well**, some of whose records copy real-well observations. Exclude it from group-based evaluation. Distinguish identical full rows from contradictory repeated-depth records, documenting all removals. Keep consecutive half-foot interval observations, which remain physically distinct even when measured values match.

In [2]:
raw=pd.read_csv(ROOT/'facies_vectors.csv')
unlabeled=pd.read_csv(ROOT/'validation_data_nofacies.csv')
assert 'Facies' in raw and 'Facies' not in unlabeled
assert set(PRIMARY_FEATURES+['PE']).issubset(raw.columns)
assert len(raw)==4149 and raw['Well Name'].nunique()==10
assert set(unlabeled['Well Name'].unique())=={'CRAWFORD','STUART'}
real=raw.loc[raw['Well Name']!='Recruit F9'].copy()
exact_dupes=int(real.duplicated().sum())
real=real.drop_duplicates().reset_index(drop=True)
# Two CROSS H CATTLE depths are duplicated with contradictory NM_M identifiers.
# Exclude BOTH entries per ambiguous depth rather than inventing an indicator value.
ambiguous_depth_mask=real.duplicated(subset=['Well Name','Depth'],keep=False)
ambiguous_rows=real.loc[ambiguous_depth_mask].copy()
ambiguous_rows.to_csv(OUT/'ambiguous_repeated_depth_rows_excluded.csv',index=False)
assert len(ambiguous_rows)==4 and ambiguous_rows['Depth'].nunique()==2
real=real.loc[~ambiguous_depth_mask].reset_index(drop=True)
assert real['Well Name'].nunique()==9 and len(real)==4064
assert not real[PRIMARY_FEATURES].isna().any().any()
well_report=real.groupby('Well Name').agg(records=('Facies','size'),valid_pe=('PE','count'),unique_classes=('Facies','nunique'))
for f in range(1,10):well_report[FACIES[f]]=real.loc[real['Facies']==f].groupby('Well Name').size().reindex(well_report.index,fill_value=0)
well_report.to_csv(OUT/'well_coverage_and_classes.csv')
audit={'original_labeled_rows':len(raw),'real_labeled_wells':real['Well Name'].nunique(),
       'pseudo_well_rows_excluded':int((raw['Well Name']=='Recruit F9').sum()),
       'exact_full_duplicates_removed':exact_dupes,
       'ambiguous_same_well_same_depth_records_removed':len(ambiguous_rows),'clean_real_records':len(real),
       'complete_pe_real_records':int(real['PE'].notna().sum()),
       'real_wells_with_pe':int(real.loc[real['PE'].notna(),'Well Name'].nunique()),
       'unlabeled_records':len(unlabeled),'unlabeled_wells':sorted(unlabeled['Well Name'].unique().tolist())}
(OUT/'data_audit.json').write_text(json.dumps(audit,indent=2))
print(json.dumps(audit,indent=2));print(well_report.to_string())
# One whole borehole, not an individual depth interval, is our independent validation unit.

{
  "original_labeled_rows": 4149,
  "real_labeled_wells": 9,
  "pseudo_well_rows_excluded": 80,
  "exact_full_duplicates_removed": 1,
  "ambiguous_same_well_same_depth_records_removed": 4,
  "clean_real_records": 4064,
  "complete_pe_real_records": 3159,
  "real_wells_with_pe": 7,
  "unlabeled_records": 830,
  "unlabeled_wells": [
    "CRAWFORD",
    "STUART"
  ]
}
                 records  valid_pe  unique_classes   SS  CSiS  FSiS  SiSh  MS  WS   D   PS  BS
Well Name                                                                                     
ALEXANDER D          466         0               8    0   117    91    44  26  69  16   98   5
CHURCHMAN BIBLE      404       404               9    8    56    51    13  30  87  34   75  50
CROSS H CATTLE       497       497               8  158   140    45    25  28  31   2   68   0
KIMZEY A             439         0               9    9    85    74    43  53  51  27   90   7
LUKE G U             461       461               7    0   117

## 2. Candidate classifiers and meaningful geological metrics

**Random Forest:** fixed 180 trees with class-weight balancing; no scaling needed. **Logistic Regression and RBF SVM:** standardize numerical features and one-hot encode NM_M inside each training fold, avoiding leakage. Inner validation holds out complete wells and selects a candidate by mean **per-well macro F1**. Neighbor-tolerant facies accuracy is secondary to exact accuracy and cannot prove geological consistency.

In [3]:
def candidate_models(features):
    numerical=[x for x in features if x!='NM_M']
    prep=ColumnTransformer([('num',StandardScaler(),numerical),
                            ('marine',OneHotEncoder(handle_unknown='ignore'),['NM_M'])]) if 'NM_M' in features else StandardScaler()
    return {
        'Logistic Regression':Pipeline([('prep',clone(prep)),('model',LogisticRegression(C=1.0,max_iter=2500,class_weight='balanced',random_state=SEED))]),
        'RBF SVM':Pipeline([('prep',clone(prep)),('model',SVC(C=2.0,gamma='scale',class_weight='balanced'))]),
        'Random Forest':RandomForestClassifier(n_estimators=180,max_depth=14,min_samples_leaf=3,
                  max_features='sqrt',class_weight='balanced_subsample',n_jobs=1,random_state=SEED)
    }

def adjacent_accuracy(true,pred):
    return float(np.mean([int(p)==int(t) or int(p) in ADJACENT[int(t)] for t,p in zip(true,pred)]))

def metrics(true,pred):
    known=sorted(np.unique(true).astype(int))
    return {'accuracy':float(accuracy_score(true,pred)),
            'macro_f1_observed':float(f1_score(true,pred,labels=known,average='macro',zero_division=0)),
            'balanced_accuracy':float(balanced_accuracy_score(true,pred)),
            'weighted_f1':float(f1_score(true,pred,average='weighted',zero_division=0)),
            'adjacent_accuracy':adjacent_accuracy(true,pred),
            'observed_facies':','.join(str(i) for i in known)}

def group_cv_select(train,features,n_splits=4):
    # Nested inner CV: each validation fold holds out COMPLETE WELLS.
    # Aggregate per-well macro F1; never evaluate current outer test well here.
    splitter=GroupKFold(n_splits=n_splits)
    results=[]
    for name,est in candidate_models(features).items():
        perwell=[]
        for itr,ival in splitter.split(train[features],train['Facies'],groups=train['Well Name']):
            model=clone(est).fit(train.iloc[itr][features],train.iloc[itr]['Facies'])
            predicted=model.predict(train.iloc[ival][features]);val=train.iloc[ival].copy()
            val['_pred']=predicted
            for well,part in val.groupby('Well Name'):
                perwell.append(metrics(part['Facies'].to_numpy(),part['_pred'].to_numpy())['macro_f1_observed'])
        results.append({'model':name,'mean_inner_well_macro_f1':float(np.mean(perwell)),
                        'inner_well_macro_f1_std':float(np.std(perwell)),'inner_val_wells':len(perwell)})
    frame=pd.DataFrame(results)
    # Deterministic predeclared tie preference: Random Forest, RBF SVM, Logistic Regression.
    preference={'Random Forest':0,'RBF SVM':1,'Logistic Regression':2}
    frame['tie_order']=frame['model'].map(preference)
    frame=frame.sort_values(['mean_inner_well_macro_f1','tie_order'],ascending=[False,True])
    return str(frame.iloc[0]['model']),frame.drop(columns='tie_order')
print('Candidates:',list(candidate_models(PRIMARY_FEATURES)))

Candidates: ['Logistic Regression', 'RBF SVM', 'Random Forest']


## 3. Proper test set: a different well every time

Outer **LeaveOneGroupOut** testing removes one real well (all its depth intervals), trains using the other eight, and repeats for all nine. In each outer fold, model selection uses **four GroupKFold splits within the eight training wells**. The held-out outer well never influences training or model selection. A fixed Random Forest is evaluated alongside the nested model-selection procedure.

In [4]:
outer=LeaveOneGroupOut()
perwell_rows=[];inner_rows=[];oof=[];rf_estimators={}
started=time.perf_counter()
for fold,(i_train,i_test) in enumerate(outer.split(real,real['Facies'],real['Well Name']),start=1):
    train=real.iloc[i_train];test=real.iloc[i_test]
    well=str(test['Well Name'].iloc[0]); assert train['Well Name'].nunique()==8 and well not in train['Well Name'].unique()
    selected,table=group_cv_select(train,PRIMARY_FEATURES,n_splits=4)
    for _,r in table.iterrows():inner_rows.append({'outer_test_well':well,**r.to_dict(),'selected':r['model']==selected})
    print(f'{fold}/9 held-out {well} ({len(test)} depth intervals); selected by inner CV: {selected}',flush=True)
    for name,est in candidate_models(PRIMARY_FEATURES).items():
        fitted=clone(est).fit(train[PRIMARY_FEATURES],train['Facies'])
        predict=fitted.predict(test[PRIMARY_FEATURES])
        row={'held_out_well':well,'outer_fold':fold,'model':name,'n_test':len(test),
             'selected_by_inner':name==selected,**metrics(test['Facies'],predict)}
        perwell_rows.append(row)
        if name=='Random Forest':rf_estimators[well]=fitted
        p=test[['Well Name','Depth','Facies']].copy();p['predicted_facies']=predict
        p['model']=name;p['outer_fold']=fold;p['inner_selected']=name==selected
        oof.append(p)
    print('    finished in',round(time.perf_counter()-started,1),'s',flush=True)
perwell=pd.DataFrame(perwell_rows)
inner_cv=pd.DataFrame(inner_rows)
out_of_fold=pd.concat(oof,ignore_index=True)
perwell.to_csv(OUT/'outer_per_well_metrics.csv',index=False)
inner_cv.to_csv(OUT/'inner_grouped_model_selection.csv',index=False)
out_of_fold.to_csv(OUT/'all_models_out_of_well_predictions.csv',index=False)
print('Elapsed evaluation seconds:',round(time.perf_counter()-started,1))
# No interval from the test well can be used to fit or select its model.

/opt/pyvenv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/opt/pyvenv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/opt/pyvenv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


/opt/pyvenv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/opt/pyvenv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/opt/pyvenv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


/opt/pyvenv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


/opt/pyvenv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/opt/pyvenv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


/opt/pyvenv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


/opt/pyvenv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/opt/pyvenv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


... (107 outputs left out)

/opt/pyvenv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


    finished in 67.3 s


Elapsed evaluation seconds: 67.4


/opt/pyvenv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


## 4. Evaluate out-of-well classification

Report equal-weighted mean per-well exact accuracy, pooled interval-weighted accuracy, macro F1 for observed facies, adjacent-facies accuracy and class confusion matrices. A random row-wise test cannot replace an unseen-well assessment because vertical depth measurements are correlated.

In [5]:
def make_confusion(actual,pred,title,path):
    cm=confusion_matrix(actual,pred,labels=list(FACIES))
    fig,ax=plt.subplots(figsize=(8,7));im=ax.imshow(cm,cmap='Blues')
    ax.set_xticks(range(9),list(FACIES.values()));ax.set_yticks(range(9),list(FACIES.values()))
    ax.set(xlabel='Predicted facies',ylabel='Actual facies',title=title)
    for r in range(9):
        for c in range(9):
            if cm[r,c]:ax.text(c,r,str(cm[r,c]),ha='center',va='center',fontsize=8,color='white' if cm[r,c]>cm.max()/2 else '#222')
    fig.colorbar(im,ax=ax,shrink=.85);fig.tight_layout();fig.savefig(path,dpi=160);plt.close(fig)
    return cm

summary=[]
for name in list(candidate_models(PRIMARY_FEATURES))+['Nested selected']:
    if name=='Nested selected':
        selected=out_of_fold[out_of_fold['inner_selected']].copy()
        pwell=perwell[perwell['selected_by_inner']].copy()
    else:
        selected=out_of_fold[out_of_fold['model']==name].copy()
        pwell=perwell[perwell['model']==name].copy()
    assert len(selected)==len(real) and selected[['Well Name','Depth','Facies']].equals(selected[['Well Name','Depth','Facies']])
    # The source contained one extra exact duplicate, removed above.
    agg=metrics(selected['Facies'].to_numpy(),selected['predicted_facies'].to_numpy())
    summary.append({'model':name,'wells':pwell['held_out_well'].nunique(),
                    'unweighted_mean_well_accuracy':float(pwell['accuracy'].mean()),
                    'sd_well_accuracy':float(pwell['accuracy'].std()),
                    'mean_well_macro_f1_observed':float(pwell['macro_f1_observed'].mean()),
                    'mean_well_adjacent_accuracy':float(pwell['adjacent_accuracy'].mean()),
                    'pooled_accuracy':agg['accuracy'],
                    'pooled_macro_f1_all_9':float(f1_score(selected['Facies'],selected['predicted_facies'],labels=list(FACIES),average='macro',zero_division=0)),
                    'pooled_adjacent_accuracy':agg['adjacent_accuracy']})
    report=classification_report(selected['Facies'],selected['predicted_facies'],labels=list(FACIES),target_names=list(FACIES.values()),output_dict=True,zero_division=0)
    pd.DataFrame([{'facies_id':k,'facies':v,**report[v]} for k,v in FACIES.items()]).to_csv(OUT/(name.lower().replace(' ','_')+'_per_class.csv'),index=False)
    make_confusion(selected['Facies'],selected['predicted_facies'],f'Out-of-well predictions — {name}',OUT/(name.lower().replace(' ','_')+'_confusion.png'))
    if name in ['Random Forest','Nested selected']:
        selected.to_csv(OUT/(name.lower().replace(' ','_')+'_oof_predictions.csv'),index=False)
summary=pd.DataFrame(summary).sort_values('unweighted_mean_well_accuracy',ascending=False)
summary.to_csv(OUT/'outer_summary.csv',index=False)
print('\nAll-well group-CV results:\n',summary.round(3).to_string(index=False))
print('\nNested model-selection counts:',perwell[perwell['selected_by_inner']]['model'].value_counts().to_dict())
print('\nFixed RF each well:\n',perwell[perwell['model']=='Random Forest'][['held_out_well','n_test','accuracy','macro_f1_observed','adjacent_accuracy','observed_facies']].round(3).to_string(index=False))
# A useful result to inspect first: errors between the nine facies classes.
from IPython.display import display, Image
display(Image(filename=str(OUT/'random_forest_confusion.png')))


All-well group-CV results:
               model  wells  unweighted_mean_well_accuracy  sd_well_accuracy  mean_well_macro_f1_observed  mean_well_adjacent_accuracy  pooled_accuracy  pooled_macro_f1_all_9  pooled_adjacent_accuracy
      Random Forest      9                          0.498             0.058                        0.403                        0.894            0.498                  0.404                     0.895
    Nested selected      9                          0.477             0.065                        0.390                        0.882            0.477                  0.390                     0.883
            RBF SVM      9                          0.470             0.054                        0.397                        0.866            0.472                  0.416                     0.867
Logistic Regression      9                          0.451             0.048                        0.381                        0.850            0.453                  0.4

## 5. Feature sensitivity: geology descriptors and missing PE

Compare **four measured logs** with **four logs + two geology descriptors** on all nine real wells. Independently, compare **six vs seven features** (adding PE) on the **same seven PE-complete wells**. Two wells entirely lack PE, so seven-well scores must not be compared uncritically with nine-well scores.

In [6]:
ablation=[]
for branch,subset,feature_sets in [
    ('nine_wells_no_pe',real,{'measured_logs_4':LOGS_4,'logs_and_geology_6':PRIMARY_FEATURES}),
    ('same_seven_pe_wells',real.dropna(subset=['PE']).copy(),{'no_pe_6':PRIMARY_FEATURES,'with_pe_7':PE_FEATURES})]:
    assert subset['Well Name'].nunique()==(9 if branch=='nine_wells_no_pe' else 7)
    for configuration,features in feature_sets.items():
        predictions=[];rows=[]
        for it,iv in LeaveOneGroupOut().split(subset,subset['Facies'],subset['Well Name']):
            a=subset.iloc[it];b=subset.iloc[iv]
            forest=clone(candidate_models(features)['Random Forest']).fit(a[features],a['Facies'])
            pred=forest.predict(b[features]);predictions.append(pd.DataFrame({'actual':b['Facies'].to_numpy(),'predicted':pred}))
            rows.append({'well':b['Well Name'].iloc[0],**metrics(b['Facies'].to_numpy(),pred)})
        combined=pd.concat(predictions,ignore_index=True);w=pd.DataFrame(rows)
        ablation.append({'experiment':branch,'configuration':configuration,'number_of_wells':len(w),
                         'n_observations':len(combined),'mean_well_accuracy':float(w['accuracy'].mean()),
                         'mean_well_macro_f1':float(w['macro_f1_observed'].mean()),
                         'pooled_accuracy':accuracy_score(combined['actual'],combined['predicted']),
                         'mean_well_adjacent_accuracy':float(w['adjacent_accuracy'].mean())})
        w.to_csv(OUT/f'ablation_{branch}_{configuration}_per_well.csv',index=False)
ablation_summary=pd.DataFrame(ablation)
ablation_summary.to_csv(OUT/'ablation_summary.csv',index=False)
print(ablation_summary.round(3).to_string(index=False))
# The PE comparison uses IDENTICAL seven PE-complete wells for fair feature ablation.

/opt/pyvenv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


/opt/pyvenv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


/opt/pyvenv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


/opt/pyvenv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


/opt/pyvenv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


/opt/pyvenv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


... (15 outputs left out)

/opt/pyvenv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


/opt/pyvenv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


         experiment      configuration  number_of_wells  n_observations  mean_well_accuracy  mean_well_macro_f1  pooled_accuracy  mean_well_adjacent_accuracy
   nine_wells_no_pe    measured_logs_4                9            4064               0.406               0.320            0.406                        0.774
   nine_wells_no_pe logs_and_geology_6                9            4064               0.498               0.403            0.498                        0.894
same_seven_pe_wells            no_pe_6                7            3159               0.483               0.390            0.483                        0.895
same_seven_pe_wells          with_pe_7                7            3159               0.491               0.409            0.491                        0.897


/opt/pyvenv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


## 6. Geological plots and interpretation

Preserve the original core-labeled, six-track log figures for **CHURCHMAN BIBLE** and **LUKE G U**, as well as per-well facies distributions. Use actual depth coordinates and a shared nine-facies color legend. Also inspect unseen-well predicted vs observed facies plots and the Random Forest's held-out permutation importances. Visual continuity is not an independent validation of rock types or Walther's Law.

In [7]:
def depth_edges(depth):
    a=np.asarray(depth,dtype=float)
    mids=(a[1:]+a[:-1])/2
    return np.r_[a[0]-(mids[0]-a[0]),mids,a[-1]+(a[-1]-mids[-1])]

def facies_plot(frame,path,title,observed=True):
    d=frame.sort_values('Depth').copy()
    assert not d['Depth'].duplicated().any(), 'Please reconcile duplicate depth rows before plotting'
    z=d['Depth'].to_numpy();edges=depth_edges(z)
    n=4 if observed else 3
    fig,ax=plt.subplots(1,n,figsize=(11 if observed else 9,10),sharey=True,
                       gridspec_kw={'width_ratios':[2,2,1,1] if observed else [2,2,1]})
    ax[0].plot(d['GR'],z,lw=.85,color='#348d52');ax[0].set_xlabel('Gamma ray');ax[0].set_ylabel('Depth (ft)')
    ax[1].plot(d['ILD_log10'],z,lw=.85,color='#a05e38');ax[1].set_xlabel('Log resistivity')
    cm=ListedColormap(COLORS);norm=BoundaryNorm(np.arange(.5,10.5,1),cm.N)
    cols=[('Observed facies','Facies'),('Predicted facies','predicted_facies')] if observed else [('Predicted facies','predicted_facies')]
    for axis,(label,key) in zip(ax[2:],cols):
        axis.pcolormesh([0,1],edges,d[key].to_numpy(dtype=int)[:,None],cmap=cm,norm=norm,shading='flat')
        axis.set_xlim(0,1);axis.set_xticks([]);axis.set_xlabel(label)
    for axis in ax:axis.grid(alpha=.2)
    ax[0].set_ylim(edges[-1],edges[0]);fig.suptitle(title)
    cb=fig.colorbar(plt.cm.ScalarMappable(cmap=cm,norm=norm),ax=ax.tolist(),ticks=range(1,10),fraction=.028,pad=.04)
    cb.ax.set_yticklabels(list(FACIES.values()))
    fig.subplots_adjust(top=.94,right=.84,left=.1,bottom=.07,wspace=.18)
    fig.savefig(path,dpi=160,bbox_inches='tight');plt.close(fig)

rf_preds=out_of_fold.loc[out_of_fold['model']=='Random Forest']
for well in ['NOLAN','CHURCHMAN BIBLE','KIMZEY A']:
    d=real[real['Well Name']==well].merge(rf_preds[['Well Name','Depth','predicted_facies']],on=['Well Name','Depth'],validate='one_to_one')
    facies_plot(d,OUT/(well.lower().replace(' ','_')+'_observed_vs_oof.png'),well+' — true vs out-of-well Random Forest',True)

importance=[]
for well in sorted(real['Well Name'].unique()):
    test=real[real['Well Name']==well]
    result=permutation_importance(rf_estimators[well],test[PRIMARY_FEATURES],test['Facies'],
              scoring='accuracy',n_repeats=4,random_state=SEED,n_jobs=1)
    importance.extend([{'well':well,'feature':feature,'mean_accuracy_drop':float(result.importances_mean[i]),
                        'sd_accuracy_drop':float(result.importances_std[i])} for i,feature in enumerate(PRIMARY_FEATURES)])
importance=pd.DataFrame(importance);importance.to_csv(OUT/'oof_permutation_importance_by_well.csv',index=False)
fig,ax=plt.subplots(figsize=(8,4.4))
p=importance.groupby('feature')['mean_accuracy_drop'].agg(['mean','std']).sort_values('mean')
ax.barh(p.index,p['mean'],xerr=p['std'],color='#6993a5');ax.axvline(0,color='black',lw=.9)
ax.set_xlabel('Mean held-out-well accuracy reduction on permutation');ax.set_title('RF permutation feature importance (nine held-out wells)')
fig.tight_layout();fig.savefig(OUT/'permutation_importance.png',dpi=160);plt.close(fig)
print(p.round(4).to_string())
# Core labels versus RF predictions from entirely held-out NOLAN.
display(Image(filename=str(OUT/'nolan_observed_vs_oof.png')))

             mean     std
feature                  
DeltaPHI   0.0237  0.0216
RELPOS     0.0301  0.0201
ILD_log10  0.0318  0.0238
GR         0.0567  0.0203
PHIND      0.0836  0.0351
NM_M       0.2059  0.0351


### 6A. Original five-log + facies visualizations (retained)

The original notebook included six-track log displays for **CHURCHMAN BIBLE** and **LUKE G U**. This version retains all five original curves (GR, ILD_log10, DeltaPHI, PHIND, PE) and the core-labeled facies track, with actual sampled depth coordinates instead of uniform-pixel image scaling. It also retains the facies-by-well frequency plots. Exploratory plots do not use model predictions or influence training.

In [ ]:
# Five measured well logs and one lithofacies track share a depth axis.
# Use 'Facies' for observed core labels or 'predicted_facies' for new-well predictions.
def plot_five_logs_and_facies(well_df, well, destination, facies_key='Facies', provenance='Core-labeled'):
    d=well_df.sort_values('Depth').copy()
    assert len(d)>=2 and not d['Depth'].duplicated().any()
    z=d['Depth'].to_numpy(dtype=float)
    edges=depth_edges(z)
    fig,ax=plt.subplots(1,6,figsize=(14,9),sharey=True,
                        gridspec_kw={'width_ratios':[1,1,1,1,1,.75]})
    styles=['#389452','#8c694d','#767676','#bf4545','#333333']
    for axis,field,color in zip(ax[:5],LOGS_5,styles):
        axis.plot(d[field].to_numpy(),z,lw=.9,color=color)
        axis.set_xlabel(field,fontsize=9)
        axis.grid(alpha=.3,lw=.4)
        axis.tick_params(axis='x',labelsize=8)
    cmap=ListedColormap(COLORS)
    norm=BoundaryNorm(np.arange(.5,10.5,1),cmap.N)
    ax[-1].pcolormesh([0,1],edges,d[facies_key].to_numpy(dtype=int)[:,None],
                      cmap=cmap,norm=norm,shading='flat')
    ax[-1].set(xlim=(0,1),xlabel='Facies',xticks=[])
    ax[0].set_ylabel('Depth (ft)')
    ax[0].set_ylim(edges[-1],edges[0])
    fig.suptitle(f'{well} | {provenance} facies and measured well logs')
    cb=fig.colorbar(plt.cm.ScalarMappable(cmap=cmap,norm=norm),ax=list(ax),
                    ticks=range(1,10),fraction=.018,pad=.02)
    cb.ax.set_yticklabels(list(FACIES.values()))
    fig.subplots_adjust(left=.07,right=.87,top=.92,bottom=.10,wspace=.12)
    fig.savefig(destination,dpi=150,bbox_inches='tight')
    plt.close(fig)

# Keep the TWO labeled wells shown in the original notebook.
for well in ['CHURCHMAN BIBLE','LUKE G U']:
    subset=real.loc[real['Well Name']==well]
    assert subset['PE'].notna().all()
    plot_five_logs_and_facies(subset,well,
                             OUT/(well.lower().replace(' ','_')+'_original_six_track.png'))

# Retain per-well and overall facies count plots, using cleaned real-well data.
counts=pd.crosstab(real['Well Name'],real['Facies']).reindex(columns=list(FACIES),fill_value=0)
counts.columns=list(FACIES.values())
fig,axs=plt.subplots(3,3,figsize=(16,10))
for axis,(well,row) in zip(axs.ravel(),counts.iterrows()):
    axis.bar(row.index,row.to_numpy(),color=COLORS,edgecolor='#555',linewidth=.4)
    axis.set_title(well,fontsize=10)
    axis.tick_params(axis='x',labelsize=8)
    axis.grid(axis='y',alpha=.15)
fig.suptitle('Facies distribution by real labeled well (cleaned)')
fig.tight_layout(rect=(0,0,1,.96))
fig.savefig(OUT/'facies_distribution_by_well.png',dpi=150)
plt.close(fig)

fig,axis=plt.subplots(figsize=(9,4))
axis.bar(list(FACIES.values()),
         real['Facies'].value_counts().reindex(list(FACIES),fill_value=0),
         color=COLORS,edgecolor='#555',linewidth=.4)
axis.set(title='Overall core-labeled facies (cleaned)',
         xlabel='Facies',ylabel='Labeled depth intervals')
axis.grid(axis='y',alpha=.2)
fig.tight_layout()
fig.savefig(OUT/'facies_distribution_overall.png',dpi=150)
plt.close(fig)

# Display the most important retained diagrams when run interactively.
from IPython.display import display, Image
for name in ['churchman_bible_original_six_track.png','luke_g_u_original_six_track.png',
             'facies_distribution_by_well.png']:
    display(Image(filename=str(OUT/name)))
print('Retained original well-log figures and facies distributions saved.')

## 7. Inference on unlabelled STUART and CRAWFORD wells

After the well-wise evaluation is complete, select a model by grouped CV across all nine labeled wells and refit it on all 4,064 cleaned rows. Then predict 830 unlabeled intervals: **STUART (474)** and **CRAWFORD (356)**. They have no core-facies ground truth: their predictions are not tested accuracy.

In [8]:
chosen_all,final_inner=group_cv_select(real,PRIMARY_FEATURES,n_splits=4)
final_inner.to_csv(OUT/'final_selection_grouped_cv.csv',index=False)
print('Final group-CV algorithm selection (all nine labeled wells):',chosen_all)
print(final_inner.round(4).to_string(index=False))
selected_model=clone(candidate_models(PRIMARY_FEATURES)[chosen_all]).fit(real[PRIMARY_FEATURES],real['Facies'])
rf_full=clone(candidate_models(PRIMARY_FEATURES)['Random Forest']).fit(real[PRIMARY_FEATURES],real['Facies'])
assert not unlabeled[PRIMARY_FEATURES].isna().any().any()
predictions=unlabeled.copy()
predictions['predicted_facies']=selected_model.predict(unlabeled[PRIMARY_FEATURES])
predictions['predicted_label']=predictions['predicted_facies'].map(FACIES)
predictions['random_forest_predicted_facies']=rf_full.predict(unlabeled[PRIMARY_FEATURES])
predictions['random_forest_predicted_label']=predictions['random_forest_predicted_facies'].map(FACIES)
predictions.to_csv(OUT/'stuart_crawford_predictions.csv',index=False)
for well in ['STUART','CRAWFORD']:
    sub=predictions[predictions['Well Name']==well]
    facies_plot(sub,OUT/(well.lower()+'_prediction.png'),well+' — '+chosen_all+' (unlabeled)',False)
    print(well,'intervals:',len(sub),'predicted class frequencies:',sub['predicted_label'].value_counts().sort_index().to_dict())
assert len(predictions)==830
# Show the full five-log petrophysical tracks for the two unlabelled prediction wells.
# A predicted facies color track is NOT a measured core label.
for well in ['STUART','CRAWFORD']:
    sub=predictions.loc[predictions['Well Name']==well]
    plot_five_logs_and_facies(sub,well,
        OUT/(well.lower()+'_predicted_six_track.png'),
        facies_key='predicted_facies',provenance='Model-predicted (unverified)')
    display(Image(filename=str(OUT/(well.lower()+'_predicted_six_track.png'))))

/opt/pyvenv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/opt/pyvenv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/opt/pyvenv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


/opt/pyvenv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/opt/pyvenv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


/opt/pyvenv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/opt/pyvenv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


/opt/pyvenv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/opt/pyvenv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


/opt/pyvenv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


/opt/pyvenv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/opt/pyvenv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


... (4 outputs left out)

/opt/pyvenv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/opt/pyvenv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


Final group-CV algorithm selection (all nine labeled wells): Random Forest
              model  mean_inner_well_macro_f1  inner_well_macro_f1_std  inner_val_wells
      Random Forest                    0.4012                   0.0543                9
            RBF SVM                    0.3905                   0.0561                9
Logistic Regression                    0.3653                   0.0571                9


STUART intervals: 474 predicted class frequencies: {'BS': 6, 'CSiS': 99, 'D': 13, 'FSiS': 72, 'MS': 22, 'PS': 134, 'SS': 4, 'SiSh': 41, 'WS': 83}


CRAWFORD intervals: 356 predicted class frequencies: {'BS': 17, 'CSiS': 69, 'D': 61, 'FSiS': 20, 'MS': 4, 'PS': 95, 'SS': 3, 'SiSh': 23, 'WS': 64}


## 8. Reproducibility and limitations

- Only nine independent labeled wells; rare facies (especially class 9) have limited well coverage.
- Geological predictors NM_M/RELPOS require independent availability in practical field applications.
- PE is excluded only from the primary six-feature model; the raw datasets and a seven-feature comparison retain PE.
- Exact accuracy is primary; adjacent-facies accuracy is a permissive neighbor metric, not independent geological proof.
- Classification outputs do not establish permeability, reserves, RQI, or verified stratigraphic correlation.
- Run all cells from the repository root to regenerate the audit CSVs, test scores, geological plots, and unlabeled well predictions.

In [9]:
report={
 'strategy':'nested leave-one-real-well-out evaluation, four-well-group-fold inner selection; no hyperparameter search',
 'seed':SEED,'n_real_labeled_wells':int(real['Well Name'].nunique()),'n_clean_rows':len(real),
 'candidate_models':list(candidate_models(PRIMARY_FEATURES)),
 'primary_fixed_rf':summary[summary['model']=='Random Forest'].iloc[0].to_dict(),
 'nested_selection':summary[summary['model']=='Nested selected'].iloc[0].to_dict(),
 'nested_selected_frequencies':perwell[perwell['selected_by_inner']]['model'].value_counts().to_dict(),
 'final_deployment_model':chosen_all,'final_cv_table':final_inner.to_dict(orient='records'),
 'unlabeled_prediction_rows':len(predictions),
 'warnings':['Cross-well transfer is much harder than row-random holdout.',
             'Unlabeled STUART and CRAWFORD cannot establish measured accuracy.',
             'Class 9 is absent from many held-out wells.']}
(OUT/'results_summary.json').write_text(json.dumps(report,indent=2,default=lambda x:x.item() if isinstance(x,np.generic) else x))
print('Saved results. Outer benchmark mean-well accuracy RF:',round(report['primary_fixed_rf']['unweighted_mean_well_accuracy'],4))
print('Nested procedure mean-well accuracy:',round(report['nested_selection']['unweighted_mean_well_accuracy'],4))

Saved results. Outer benchmark mean-well accuracy RF: 0.4983
Nested procedure mean-well accuracy: 0.4767
